# Adaptive functional GNN (robust4) — Kaggle T4

Runs the staged nested-LOSO search for `AdaptiveFuncGNN` on the frozen 794-subject /
17-site dual cohort.

## Before you run — notebook settings (right-hand panel)

1. **Accelerator → GPU T4 ×2** (one GPU is used; the model is small)
2. **Internet → On** (needed for `git clone` and `pip install`)
3. **Add Data →** attach your private dataset containing:
   - `cohort_final.csv`
   - `fc_z.npy`   (~39 MB)
   
   Flat or nested layout both work — the staging cell finds them.

`robust4` node features are recomputed from `fc_z` in-script (~12 s), so there is
no third file to upload.

## Expected runtime on T4

| Phase | ~time |
|---|---|
| Stage A — sweep `k ∈ {5,10,20,30}` + final fits | ~1.5 h |
| Stage B — sweep `layers{2,3,4} × hidden{64,128}` + final | ~2 h |
| **Both** | **~3.5 h** |

Fits inside one 12 h session. Checkpointing is still on — see the last cell for
how to resume if a session is killed.

In [ ]:
# 1. Confirm the GPU
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2. Get the code
!rm -rf /kaggle/working/Thesis-with-ABIDE-dataset
!git clone -q https://github.com/hazrotali2003028/Thesis-with-ABIDE-dataset.git /kaggle/working/Thesis-with-ABIDE-dataset
%cd /kaggle/working/Thesis-with-ABIDE-dataset
!git log --oneline -1

In [ ]:
# 3. Dependencies. torch + CUDA are preinstalled on Kaggle.
#    bctpy is needed for robust4; nolds is imported at module level in
#    node_features.py, so install it too even though robust4 does not use it.
!pip -q install torch_geometric bctpy nolds
import numpy, torch
print('numpy', numpy.__version__, '| torch still OK:', torch.cuda.is_available())

In [ ]:
# 4. Stage the uploaded data into the layout the script expects:
#      <data-dir>/cohort_final.csv  and  <data-dir>/cache/fc_z.npy
import glob, os, shutil

DATA = '/kaggle/working/data'
os.makedirs(os.path.join(DATA, 'cache'), exist_ok=True)

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'{name} not found under /kaggle/input — did you attach the dataset?'
    return hits[0]

shutil.copy(find('cohort_final.csv'), f'{DATA}/cohort_final.csv')
shutil.copy(find('fc_z.npy'), f'{DATA}/cache/fc_z.npy')

import pandas as pd, numpy as np
print('cohort', pd.read_csv(f'{DATA}/cohort_final.csv').shape,
      '| fc_z', np.load(f'{DATA}/cache/fc_z.npy').shape)

In [ ]:
# 5. RESUME (no-op on a first run).
#    If you attached a previous run's notebook output, restore its result CSV and
#    inner-HP cache so finished (site, seed) rows are skipped.
import glob, shutil, os
restored = 0
for pat in ('adaptive_gnn_stage*.csv', 'adaptive_gnn_stage*_hp.json'):
    for f in glob.glob(f'/kaggle/input/**/{pat}', recursive=True):
        shutil.copy(f, '/kaggle/working/'); restored += 1
        print('restored', os.path.basename(f))
print('nothing to resume — fresh run' if not restored else f'{restored} file(s) restored')

## Stage A — sweep `k` (the new τ)

`k ∈ {5, 10, 20, 30}` at `layers=2, hidden=64`. This sweep is mandatory: pinning the
neighbourhood size would repeat the exact error that produced the τ=0.3 /
density-0.75 null.

Smoke-test first (2 sites, 1 seed, 6 epochs — about a minute) to confirm the data
mounted and the adaptive-topology unit test passes.

In [ ]:
!python dualgraph/train_adaptive_gnn.py --stage a --smoke \
    --data-dir /kaggle/working/data --out-dir /kaggle/working

In [ ]:
# Full Stage A (~1.5 h on T4). Re-runnable: finished rows are skipped.
!python dualgraph/train_adaptive_gnn.py --stage a \
    --data-dir /kaggle/working/data --out-dir /kaggle/working

In [ ]:
# Which k did the inner CV pick most often? -> feeds Stage B
import pandas as pd, json, collections
d = pd.read_csv('/kaggle/working/adaptive_gnn_stagea.csv')
ks = [json.loads(h)['k'] for h in d.hp]
cnt = collections.Counter(ks)
BEST_K = cnt.most_common(1)[0][0]
print('k selection counts:', dict(cnt), '-> BEST_K =', BEST_K)
print('Stage A honest LOSO ROC =',
      round(d.groupby('site').honest_roc_auc.mean().mean(), 4))

## Stage B — capacity sweep at the winning `k`

`layers ∈ {2,3,4} × hidden ∈ {64,128}`. This tests the oversmoothing hypothesis
rather than assuming it: on a 111-node graph at k≈10, two hops already reach the
whole graph, so deeper layers may only re-mix and collapse the pooled vector.
Watch `pooled_sd` — if it drops sharply at `layers=4`, that is oversmoothing
measured directly, which is far stronger evidence than a noisy 17-site AUC.

In [ ]:
# Full Stage B (~2 h on T4)
!python dualgraph/train_adaptive_gnn.py --stage b --best-k {BEST_K} \
    --data-dir /kaggle/working/data --out-dir /kaggle/working

## Results

The **honest** columns are the reportable result. The `paper1_*` columns use
Paper 1's epoch-selection rule (epoch chosen on the test site) and are for
comparison only — measured on this dataset that rule inflates AUC by **+0.100
(p = 1.9e-06)**, so it reads the answer key and must never be reported as a result.
`inflation = paper1 − honest` quantifies the gap per site.

In [ ]:
import sys, glob, pandas as pd
sys.path.insert(0, '/kaggle/working/Thesis-with-ABIDE-dataset/dualgraph')
from train_adaptive_gnn import report

# report() prints, for each stage: the HONEST per-site table, the PAPER1 per-site
# table (same metric set), and the per-metric inflation (paper1 - honest).
for f in sorted(glob.glob('/kaggle/working/adaptive_gnn_stage*.csv')):
    print('=' * 78); print(f)
    report(pd.read_csv(f))

print('\nanchors:  edge-SVM 0b = 0.658 | robust4 flat SVM = 0.630 | motion floor = 0.561')
print('the question is not "beat 0.658" (both channels derive from fc_z, so that is')
print('the ceiling) but "does message passing beat 0.630, the same features flat?"')

In [ ]:
# Depth vs oversmoothing (Stage B only)
import pandas as pd, json, os
p = '/kaggle/working/adaptive_gnn_stageb.csv'
if os.path.exists(p):
    d = pd.read_csv(p)
    d['layers'] = [json.loads(h)['layers'] for h in d.hp]
    d['hidden'] = [json.loads(h)['hidden'] for h in d.hp]
    print(d.groupby(['layers','hidden'])[['honest_roc_auc','pooled_sd']]
           .mean().round(4).to_string())
else:
    print('run Stage B first')

## If the session is killed before finishing

1. **Save Version** (commit) so `/kaggle/working` becomes the notebook's output.
2. In a new run: **Add Data → Notebook Output →** attach this notebook's previous output.
3. Run the cells from the top. Cell 5 restores `adaptive_gnn_stage*.csv` and the
   inner-HP cache, and the script skips every `(site, seed)` already finished —
   including the expensive inner HP search, which is cached per site.

Quota note: Kaggle allows 30 GPU-hours/week; the full search is ~3.5 h.